In [0]:
dbutils.widgets.text("config_file_path", "")

config_file_path = dbutils.widgets.get("config_file_path")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sha2 , col, current_date , year

In [0]:
%run ./notebook1

Config loaded successfully
Input Path: /Volumes/csvfiles/default/demovol
Bronze Table: csvfiles.default.bronze_table


In [0]:
# /Volumes/csvfiles/default/demovol/pipeline_config.json
# print(config_file_path)

/Volumes/csvfiles/default/demovol/pipeline_config.json


In [0]:
#import files

df = spark.read.format("csv") \
    .option("header", "true") \
    .load(input_path+"/EmployeeRawData.csv")


# df.show()



+------------+----------+---------+--------------------+-----------+----------+----------+-----------+------------+
|Employee ID0|First Name|Last Name|               Email| Department|    Salary| Hire Date|        SSN|Employee ID8|
+------------+----------+---------+--------------------+-----------+----------+----------+-----------+------------+
|      EMP001|      John|      Doe|john.doe@example.com|Engineering|$85,000.00|2022-01-15|123-45-6789|      EMP001|
|      EMP002|      Jane|    Smith|jane.smith@exampl...|  Marketing|$72,000.00|2021-11-01|234-56-7890|      EMP002|
|      EMP003|   Michael|  Johnson|michael.j@example...|      Sales|$68,000.00|2023-03-10|345-67-8901|      EMP003|
|      EMP004|     Emily|    Davis|emily.davis@examp...|         HR|$65,000.00|2020-07-22|456-78-9012|      EMP004|
|      EMP005|     David|   Wilson|david.wilson@exam...|Engineering|$92,000.00|2019-05-18|567-89-0123|      EMP005|
|      EMP006|     Sarah|    Brown| sarah.b@example.com|    Finance|$78,

In [0]:
from pyspark.sql.functions import col

# The CSV has duplicate 'customer_id' column at positions 0 and 9
# print("Original columns-------->", df.columns)

# Drop the duplicate customer_id col.
df_cleaned = df.drop('Employee ID8')
# rename the first col.
df_cleaned = df_cleaned.withColumnRenamed('Employee ID0', 'Employee_ID')

# print("remove duplicates--------->", df_cleaned.columns)
# df_cleaned.show(5)
df_cleaned_channel = df_cleaned.toDF(*[c.strip().replace(" ", "_") for c in df_cleaned.columns])

# display(df_cleaned_channel)

Original columns--------> ['Employee ID0', 'First Name', 'Last Name', 'Email', 'Department', 'Salary', 'Hire Date', 'SSN', 'Employee ID8']
remove duplicates---------> ['Employee_ID', 'First Name', 'Last Name', 'Email', 'Department', 'Salary', 'Hire Date', 'SSN']


Employee_ID,First_Name,Last_Name,Email,Department,Salary,Hire_Date,SSN
EMP001,John,Doe,john.doe@example.com,Engineering,"$85,000.00",2022-01-15,123-45-6789
EMP002,Jane,Smith,jane.smith@example.com,Marketing,"$72,000.00",2021-11-01,234-56-7890
EMP003,Michael,Johnson,michael.j@example.com,Sales,"$68,000.00",2023-03-10,345-67-8901
EMP004,Emily,Davis,emily.davis@example.com,HR,"$65,000.00",2020-07-22,456-78-9012
EMP005,David,Wilson,david.wilson@example.com,Engineering,"$92,000.00",2019-05-18,567-89-0123
EMP006,Sarah,Brown,sarah.b@example.com,Finance,"$78,000.00",2022-09-05,678-90-1234
EMP007,James,Taylor,james.t@example.com,Engineering,"$88,000.00",2021-02-28,789-01-2345
EMP008,Jessica,Anderson,jessica.a@example.com,Marketing,"$75,000.00",2023-01-12,890-12-3456
EMP009,William,Thomas,william.t@example.com,Sales,"$71,000.00",2020-11-20,901-23-4567
EMP010,Ashley,Jackson,ashley.j@example.com,HR,"$62,000.00",2022-04-14,012-34-5678


In [0]:
# add encryption

df_cleaned_encrypted = df_cleaned_channel
for col_name in config["sensitive_columns"]:
    if col_name in df_cleaned_encrypted.columns:
        df_cleaned_encrypted = df_cleaned_encrypted.withColumn(col_name+"_encrypted", sha2(col(col_name).cast("string"), 256))

# display(df_cleaned_encrypted)

Employee_ID,First_Name,Last_Name,Email,Department,Salary,Hire_Date,SSN,SSN_encrypted,Salary_encrypted
EMP001,John,Doe,john.doe@example.com,Engineering,"$85,000.00",2022-01-15,123-45-6789,01a54629efb952287e554eb23ef69c52097a75aecc0e3a93ca0855ab6d7a31a0,c08e7a2f63f203781ccfc863b16731b32563ae9b3186cd6062c518e64addf523
EMP002,Jane,Smith,jane.smith@example.com,Marketing,"$72,000.00",2021-11-01,234-56-7890,2a87a12699cef0854f5c726f45ced690ba58fc9eef3e72610725a9b26479de42,841c756dd0d6ad0204492278045827b2bd71918d0d798dc27acfffe737385f51
EMP003,Michael,Johnson,michael.j@example.com,Sales,"$68,000.00",2023-03-10,345-67-8901,0159e3ba838b89a0a4bdb66e76c1d269f40e77ceec617d85a14156ac04b7c090,deec7508debf45a898aabd88b15304126f423bc09d22eef998520633faec0d3d
EMP004,Emily,Davis,emily.davis@example.com,HR,"$65,000.00",2020-07-22,456-78-9012,34450d3629c8ff56fcb8bf40ced5f8395f73a1486d9a9796afa66c4cecabc2f3,927727bfbca513547a97694ee63ce24d9b5db68631577e9971cafe555634bac7
EMP005,David,Wilson,david.wilson@example.com,Engineering,"$92,000.00",2019-05-18,567-89-0123,21d6df2de747e93f696a81447ff6cd90a9c3bd2ad77ab7e5e1e7bc55c6ebc9bb,7028699272ac54450274c7ac2bf611af55953b7c3defe955bef077c6bcf74e76
EMP006,Sarah,Brown,sarah.b@example.com,Finance,"$78,000.00",2022-09-05,678-90-1234,ba4ae52efcaa4eccd489109cb6a7c231a41f045c504512d26fb169058746c68c,a2a239b2e15e1b283bda38ca3d4fd554c76fb10227bdbf203298a31d28772e42
EMP007,James,Taylor,james.t@example.com,Engineering,"$88,000.00",2021-02-28,789-01-2345,1ecdb8649cfa0aa0b2607ebcfb282360312884153030d41b0605a20b55182845,19d47b70a2e540349d353062a310df4163daff2971edf9d0cc0b16eb5d3ea87c
EMP008,Jessica,Anderson,jessica.a@example.com,Marketing,"$75,000.00",2023-01-12,890-12-3456,5144e522dddd37d151df5328b9cb42c6addfab30bc1e42501c6e153bbfb43ad0,32abe77179b0106aa718c92b3ae0d64f276ca768732735495276c74fa37b8957
EMP009,William,Thomas,william.t@example.com,Sales,"$71,000.00",2020-11-20,901-23-4567,1957cb6380193f4c39614053e8b4ced1e424389a6de18b88be2fafa5c6835c62,352a11ee98addd922b3fad66552a9521caed7fc8360f3d79c6401e0b6303c7bf
EMP010,Ashley,Jackson,ashley.j@example.com,HR,"$62,000.00",2022-04-14,012-34-5678,af7d09747c8cbe325850a962be1af48ae6ddf23c98476caaa137a29be46562f0,20dd120a7502f16300940eda4f4d4fbdcf860ae54f214921e5133551d61188d5


In [0]:
# # Add partition column
df_partition = df_cleaned_encrypted.withColumn("ingestion_date", current_date()) \
       .withColumn("hire_year", year("Hire_Date"))

# df_partition.display()



Employee_ID,First_Name,Last_Name,Email,Department,Salary,Hire_Date,SSN,SSN_encrypted,Salary_encrypted,ingestion_date,hire_year
EMP001,John,Doe,john.doe@example.com,Engineering,"$85,000.00",2022-01-15,123-45-6789,01a54629efb952287e554eb23ef69c52097a75aecc0e3a93ca0855ab6d7a31a0,c08e7a2f63f203781ccfc863b16731b32563ae9b3186cd6062c518e64addf523,2026-05-01,2022
EMP002,Jane,Smith,jane.smith@example.com,Marketing,"$72,000.00",2021-11-01,234-56-7890,2a87a12699cef0854f5c726f45ced690ba58fc9eef3e72610725a9b26479de42,841c756dd0d6ad0204492278045827b2bd71918d0d798dc27acfffe737385f51,2026-05-01,2021
EMP003,Michael,Johnson,michael.j@example.com,Sales,"$68,000.00",2023-03-10,345-67-8901,0159e3ba838b89a0a4bdb66e76c1d269f40e77ceec617d85a14156ac04b7c090,deec7508debf45a898aabd88b15304126f423bc09d22eef998520633faec0d3d,2026-05-01,2023
EMP004,Emily,Davis,emily.davis@example.com,HR,"$65,000.00",2020-07-22,456-78-9012,34450d3629c8ff56fcb8bf40ced5f8395f73a1486d9a9796afa66c4cecabc2f3,927727bfbca513547a97694ee63ce24d9b5db68631577e9971cafe555634bac7,2026-05-01,2020
EMP005,David,Wilson,david.wilson@example.com,Engineering,"$92,000.00",2019-05-18,567-89-0123,21d6df2de747e93f696a81447ff6cd90a9c3bd2ad77ab7e5e1e7bc55c6ebc9bb,7028699272ac54450274c7ac2bf611af55953b7c3defe955bef077c6bcf74e76,2026-05-01,2019
EMP006,Sarah,Brown,sarah.b@example.com,Finance,"$78,000.00",2022-09-05,678-90-1234,ba4ae52efcaa4eccd489109cb6a7c231a41f045c504512d26fb169058746c68c,a2a239b2e15e1b283bda38ca3d4fd554c76fb10227bdbf203298a31d28772e42,2026-05-01,2022
EMP007,James,Taylor,james.t@example.com,Engineering,"$88,000.00",2021-02-28,789-01-2345,1ecdb8649cfa0aa0b2607ebcfb282360312884153030d41b0605a20b55182845,19d47b70a2e540349d353062a310df4163daff2971edf9d0cc0b16eb5d3ea87c,2026-05-01,2021
EMP008,Jessica,Anderson,jessica.a@example.com,Marketing,"$75,000.00",2023-01-12,890-12-3456,5144e522dddd37d151df5328b9cb42c6addfab30bc1e42501c6e153bbfb43ad0,32abe77179b0106aa718c92b3ae0d64f276ca768732735495276c74fa37b8957,2026-05-01,2023
EMP009,William,Thomas,william.t@example.com,Sales,"$71,000.00",2020-11-20,901-23-4567,1957cb6380193f4c39614053e8b4ced1e424389a6de18b88be2fafa5c6835c62,352a11ee98addd922b3fad66552a9521caed7fc8360f3d79c6401e0b6303c7bf,2026-05-01,2020
EMP010,Ashley,Jackson,ashley.j@example.com,HR,"$62,000.00",2022-04-14,012-34-5678,af7d09747c8cbe325850a962be1af48ae6ddf23c98476caaa137a29be46562f0,20dd120a7502f16300940eda4f4d4fbdcf860ae54f214921e5133551d61188d5,2026-05-01,2022


In [0]:
df_partition.write.mode("overwrite") \
    .partitionBy(config["partition_columns"]["bronze"]) \
    .saveAsTable(bronze_table)

In [0]:
data = [(i,) for i in range(200)]
df = spark.createDataFrame(data, ["id"]) # transformations (logic)

display(df) #action
# df.show() # action  (Lazy evaluation)  (directed ascyclic graph)

id
0
1
2
3
4
5
6
7
8
9
